# Stufe 1 — Layout-Erkennung und Layout-Schema

**KI-gestützte Dokumentenaufbereitung · Referenz-Implementierung · Niveau DQR 5/6**

Dieses Notebook baut die erste Stufe der Pipeline: aus einer beliebigen PDF-Seite
wird eine typisierte, verortete und geordnete Liste von Layout-Bausteinen.

## Einordnung ins Phasenmodell

| Phase | Inhalt | in diesem Notebook |
|---|---|---|
| 00 | Triage & Klassifikation | teilweise — Seiteninventar |
| **01b** | **Layout-Analyse** | **Schwerpunkt** |
| 02 | Datenmodellierung | **Schwerpunkt** — das Zwischenschema |
| 03–08 | Prompting bis Monitoring | spätere Stufen |

## Lernziele

Nach dem Durcharbeiten können Sie
1. erklären, warum Layout-Analyse **nicht** in ein generelles Vision-LLM gehört,
2. die Vorverarbeitung eines Detektors aus seiner Konfigurationsdatei rekonstruieren,
3. Bounding Boxes zwischen vier Koordinatensystemen fehlerfrei umrechnen,
4. ein Zwischenschema entwerfen, das Detektionswissen ohne Informationsverlust hält,
5. begründen, an welchen Stellen eine Abbildung auf ein Standardformat verlustbehaftet ist.

## Werkzeuge

| Stufe | Werkzeug | Läuft |
|---|---|---|
| Rendern | PyMuPDF | überall |
| **Layout** | **PP-DocLayoutV3 (ONNX) + ONNX Runtime** | Windows/macOS/Linux nativ |
| Erkennung | PaddleOCR-VL-1.5 GGUF in LM Studio | überall |
| Semantik | Gemma 4 12B in LM Studio | überall |

Kein Docker, kein WSL, kein PaddlePaddle, kein vLLM.

> **Benötigte Modelldateien.** Dieses Notebook arbeitet mit dem ONNX-Export von
> `phungpx/PP-DocLayoutV3-ONNX`. Legen Sie `pp_doclayoutv3.onnx` (134 MB) und
> `labels.json` in den Ordner `models/` neben dieses Notebook. Andere Exporte
> desselben Modells haben **andere Ein- und Ausgänge** — siehe Abschnitt 7.

---
## 0. Warum ein eigener Detektor statt eines großen VLM?

Die naheliegende Idee lautet: ein Vision-LLM sieht die Seite an und beschreibt ihren
Aufbau. Die Entwickler:innen von PaddleOCR-VL haben sich bewusst dagegen entschieden.
Ihre Begründung: End-to-End-Ansätze auf VLM-Basis arbeiten mit langen autoregressiven
Sequenzen. Das erzeugt hohe Latenz und Speicherverbrauch — und erhöht das Risiko
**instabiler Layout-Analyse und von Halluzinationen**, besonders ausgeprägt bei
mehrspaltigen oder gemischt text-grafischen Layouts.

Deshalb steht vor dem Erkennungsmodell ein eigenes, kleines Detektionsmodell:

```
PP-DocLayoutV3   (DETR-artig, 300 Queries, ~134 MB als ONNX)
   Eingang
   └── pixel_values   (B, 3, 800, 800)   RGB, 0…1
   Ausgänge
   ├── logits         (B, 300, 25)       Klassenscores, sigmoid
   ├── pred_boxes     (B, 300,  4)       cxcywh, normiert 0…1
   ├── order_logits   (B, 300, 300)      Lesereihenfolge als Zeigermatrix
   └── out_masks      (B, 300, 200, 200) Instanzmasken, Stride 4
```

Drei Eigenschaften sind für uns entscheidend:

- Es **erkennt keinen Text**. Es lokalisiert, klassifiziert und ordnet. Genau unsere Stufe 1.
- Die Lesereihenfolge kommt als **Matrix über Blockpaare**, nicht als Index. Damit gibt
  es eine Konfidenz *pro Paar* — ein Gütemaß, ohne ein Modell nach seiner Sicherheit
  fragen zu müssen (die wäre ohnehin kaum kalibriert).
- Es ist ein Detektor, keine Generierung: Millisekunden statt Sekunden pro Seite.

> ⚠️ Dass diese vier Köpfe sichtbar sind, ist **keine Eigenschaft des Modells, sondern
> des Exports**. Andere ONNX-Exporte desselben Gewichtsstands backen die
> Nachverarbeitung in den Graphen und liefern nur noch eine fertige Tabelle
> `(300, 7)` mit einem ganzzahligen Rang statt einer Matrix. Bequemer — aber die
> Konfidenz pro Paar und die Masken sind dann weg.

---
## 1. Vorbereitung

Die Pfade unten sind auf drei reale Testseiten gesetzt — bewusst keine sauberen
Beispielseiten, sondern Layout-Problemfälle. Passen Sie sie an Ihre Umgebung an.

Die Abhängigkeiten werden **im Terminal** installiert, nicht in der Zelle: ein
`%pip install` mitten im Notebook erzwingt einen Kernel-Neustart und macht den
Zustand der Sitzung undurchsichtig.

```bash
pip install pymupdf pdfplumber pydantic opencv-python-headless numpy docling-core onnxruntime
```

In [ ]:
from __future__ import annotations

import json
from enum import Enum
from pathlib import Path
from typing import Literal

import cv2
import numpy as np
import onnxruntime as ort
import pymupdf
import time
from pydantic import BaseModel, Field, model_validator

UPLOADS = Path("uploads")          # ANPASSEN
SEITEN = {
    "zimbardo":     UPLOADS / "Zimbardo_Psychologie_18Aufl_Extracted.pdf",
    "wahrnehmung":  UPLOADS / "Wahrnehmungspsychologie_Extracted.pdf",
    "tietze":       UPLOADS / "HalbleiterSchaltungstechnik_TietzeSchenk_2002_Extracted.pdf",
}

MODELL_DIR = Path("models")                          # Repo liegt neben dem Notebook
ONNX_DATEI = MODELL_DIR / "pp_doclayoutv3.onnx"
RENDER_DPI = 200          # Standard; für den Scan siehe Abschnitt 3

print(f"onnxruntime {ort.__version__}   Provider: {ort.get_available_providers()}\n")
for name, pfad in SEITEN.items():
    print(f"{name:12s} {'vorhanden' if pfad.exists() else 'FEHLT':10s} {pfad}")
print(f"{'modell':12s} {'vorhanden' if ONNX_DATEI.exists() else 'FEHLT':10s} {ONNX_DATEI}")

---
## 2. Die drei Testseiten kennen

Bevor irgendein Modell läuft: Was liegt überhaupt vor? Die folgende Zelle ist
Phase 00 im Kleinen — Triage. Achten Sie besonders auf die letzte Spalte.

In [ ]:
def inventar(pfad: Path) -> dict:
    """Grobinventar einer Seite - Phase 00."""
    dok = pymupdf.open(pfad)
    seite = dok[0]
    bloecke = seite.get_text("dict")["blocks"]
    schriften = {
        s["font"]
        for b in bloecke if b["type"] == 0
        for z in b["lines"] for s in z["spans"]
    }
    bilder = seite.get_images(full=True)
    aufloesung = None
    if bilder:
        info = dok.extract_image(bilder[0][0])
        aufloesung = f"{info['width']}x{info['height']} ({info['ext']}, {info.get('bpc')} bpc)"
    return {
        "Seitengröße pt": f"{seite.rect.width:.0f} x {seite.rect.height:.0f}",
        "Rotation": seite.rotation,
        "Textblöcke": sum(1 for b in bloecke if b["type"] == 0),
        "Bilder": len(bilder),
        "Vektorobjekte": len(seite.get_drawings()),
        "Textlänge": len(seite.get_text()),
        "Schriften": ", ".join(sorted(schriften)[:3]) or "—",
        "Bildauflösung": aufloesung or "—",
    }

for name, pfad in SEITEN.items():
    if not pfad.exists():
        continue
    print(f"\n=== {name} ===")
    for k, v in inventar(pfad).items():
        print(f"  {k:16s} {v}")

### Was das Inventar verrät

**Zimbardo** und **Wahrnehmungspsychologie** sind born-digital: Originalschriften
(MinionPro, MyriadPro, Frutiger, Melior), viele Vektorobjekte, kein oder ein Bild.

**Tietze-Schenk** ist ein 1-Bit-Scan — und hat trotzdem einen Textlayer.
Die Schriften heißen dort *Helvetica* und *Times-Roman*: Substitutionsschriften,
also ein nachträglich eingelegter OCR-Layer.

> ⚠️ **Die naive Triage-Regel scheitert hier.**
> Die verbreitete Heuristik „`page.get_text()` ist leer ⇒ Scan" stuft diese Seite
> als born-digital ein und schickt sie in den CPU-Pfad. Der Textlayer ist aber
> fehlerhaft (`10 !2` statt `10 Ω`, `I00` statt `100`) und hat **sämtliche
> abgesetzten Formeln stillschweigend verworfen**.
>
> **Regel:** Ein vorhandener Textlayer ist keine Qualitätsaussage. Er muss selbst
> validiert werden, sonst verankert man Ergebnisse an einer Lüge.

---
## 3. Die Vorverarbeitung rekonstruieren

Ein Detektor ist nur so gut wie seine Vorverarbeitung. Wer sie falsch nachbaut,
bekommt systematisch verschobene oder verschlechterte Ergebnisse — ein Fehler, der
auf jeder Seite gleich falsch ist und optisch *fast* plausibel aussieht.

Für PP-DocLayoutV3 gibt es **zwei Quellen**, die dasselbe sagen:

Die PaddleX-Konfiguration (`config.json` bzw. `inference.yml` des Original-Modells):

```yaml
Preprocess:
- interp: 2
  keep_ratio: false
  target_size: [800, 800]
  type: Resize
- mean: [0.0, 0.0, 0.0]
  norm_type: none
  std: [1.0, 1.0, 1.0]
  type: NormalizeImage
- type: Permute
```

Und die Referenz-Implementierung des Exports (`pp_doclayout_v3_onnx.py`), portiert
aus dem HF-`PPDocLayoutV3ImageProcessor`: `INTER_CUBIC`, `RESCALE_FACTOR = 1/255`,
`mean=0`, `std=1`.

| Eintrag | Bedeutung |
|---|---|
| `interp: 2` | `cv2.INTER_CUBIC` |
| `keep_ratio: false` | **anisotrope Stauchung**, kein Padding, kein Versatz |
| `target_size: [800, 800]` | quadratische Eingabe, unabhängig vom Seitenformat |
| `mean 0 / std 1 / norm_type none` | die Normalisierung ist ein **No-Op** |
| `Permute` | HWC → CHW |

### Die Falle: `is_scale`

Der Parameter, der entscheidet, ob durch 255 geteilt wird, steht in **keiner** der
beiden Dateien. In PaddleDetection ist `is_scale=True` der Default, und die
Referenz-Implementierung teilt tatsächlich durch 255. **Wertebereich also 0…1, ohne
ImageNet-Statistik.**

Das ist kein Detail: Andere Beispielskripte im Umlauf ziehen zusätzlich
`mean=[0.485, 0.456, 0.406]` ab und teilen durch `std=[0.229, 0.224, 0.225]` — obwohl
die mitgelieferte Konfigurationsdatei desselben Repos ausdrücklich `norm_type: none`
sagt. Abschnitt 7 misst nach, statt zu glauben.

> 💡 **Lehrpunkt:** Wenn Konfigurationsdatei und Beispielcode sich widersprechen,
> gewinnt die Konfigurationsdatei — sie beschreibt das Training, der Beispielcode
> nur die Gewohnheit der Person, die ihn geschrieben hat.

In [ ]:
ZIEL = (800, 800)            # (Breite, Höhe) - für cv2.resize
INTERP = cv2.INTER_CUBIC     # entspricht interp: 2
SKALIERE_AUF_EINS = True     # is_scale - siehe Abschnitt 3 und die Messung in Abschnitt 7


def vorverarbeiten(bild_rgb: np.ndarray, skaliere_auf_eins: bool = SKALIERE_AUF_EINS) -> np.ndarray:
    """Seitenbild (RGB, HxWx3, uint8) -> Modelltensor NCHW float32.

    Der Export hat genau EINEN Eingang, `pixel_values`. Es gibt kein `scale_factor`
    und kein `im_shape`: die Rücktransformation der Boxen passiert bei uns, nicht
    im Graphen. Siehe Abschnitt 7.
    """
    if bild_rgb.ndim != 3 or bild_rgb.shape[2] != 3:
        raise ValueError(f"Erwarte HxWx3, bekommen {bild_rgb.shape}")

    klein = cv2.resize(bild_rgb, ZIEL, interpolation=INTERP)    # 1. Resize
    tensor = klein.astype(np.float32)
    if skaliere_auf_eins:
        tensor /= 255.0                                         # 2. is_scale
    tensor = np.transpose(tensor, (2, 0, 1))[None, ...]         # 3. Permute + Batch
    return np.ascontiguousarray(tensor)


def seite_rendern(pfad: Path, dpi: int = RENDER_DPI, seite: int = 0):
    """PDF-Seite -> (RGB-Bild als uint8-Array, pymupdf-Seitenobjekt)."""
    dok = pymupdf.open(pfad)
    pg = dok[seite]
    pix = pg.get_pixmap(dpi=dpi)
    bild = np.frombuffer(pix.samples, dtype=np.uint8)
    bild = bild.reshape(pix.height, pix.width, pix.n)[:, :, :3]
    return np.ascontiguousarray(bild), pg


bild, pg = seite_rendern(SEITEN["zimbardo"])
tensor = vorverarbeiten(bild)
print("Seite   ", f"{pg.rect.width:.0f} x {pg.rect.height:.0f} pt")
print("Bild    ", f"{bild.shape[1]} x {bild.shape[0]} px bei {RENDER_DPI} dpi")
print("Tensor  ", tensor.shape, tensor.dtype,
      f"Wertebereich {tensor.min():.3f}..{tensor.max():.3f}")

### `keep_ratio: false` — die Stauchung verstehen

Das ist der wichtigste Satz der ganzen Konfigurationsdatei. Das Modell sieht **immer**
ein Quadrat. Eine A4-Seite im Verhältnis 3:4 wird horizontal um Faktor 4/3 gestreckt
in das Quadrat gepresst.

Zwei Konsequenzen, eine gute und eine unangenehme:

✅ **Die Rücktransformation ist trivial.** Ohne Padding gibt es keinen Versatz — nur
eine achsenweise Streckung mit *unterschiedlichen* Faktoren für x und y.

⚠️ **Die Rendering-Auflösung ist für Stufe 1 gleichgültig.** Ob Sie mit 150 oder
300 dpi rendern: das Bild landet in 800×800. Die dpi zählt erst für die
**Ausschnitte**, die in Stufe 3 an das Erkennungsmodell gehen.

> 💡 **Faustregel:** Hoch rendern, Layout auf der verkleinerten Kopie, Ausschnitte
> aus dem Original. Für den Tietze-Schenk-Scan bei seinen nativen ~299 dpi bleiben,
> statt hochzuskalieren — Hochskalieren erfindet keine Information.

In [ ]:
for name, pfad in SEITEN.items():
    if not pfad.exists():
        continue
    bild_i, _ = seite_rendern(pfad)
    h, b = bild_i.shape[:2]
    fx, fy = 800 / b, 800 / h
    print(f"{name:12s} {b:5d} x {h:5d} px  ->  800 x 800   "
          f"Stauchung x={fx:.3f}  y={fy:.3f}   "
          f"Verzerrung {fx / fy:.3f}")

Eine Verzerrung von 1,0 hieße formtreu. Alle drei Seiten liegen darüber oder darunter —
das Modell ist darauf trainiert, das auszuhalten, aber es erklärt, warum sehr schmale
Elemente (einzeilige Randnotizen, Kolumnentitel am Blattrand) Kandidaten für Aussetzer
sind. Genau die gehören später ins Goldset.

---
## 4. Vier Koordinatensysteme

Die häufigste stille Fehlerquelle der ganzen Pipeline. In diesem Projekt treffen
**vier** Konventionen aufeinander:

| Rahmen | Einheit | Ursprung | Herkunft |
|---|---|---|---|
| `MODELL_800` | 0…800 | oben links | Detektorraum |
| `BILD_PIXEL` | Pixel | oben links | gerendertes Seitenbild |
| `SEITE_PUNKT` | pt (1/72″) | oben links (PyMuPDF) | PDF-Objekte |
| `NORMIERT_1000` | 0…1000, `[y0,x0,y1,x1]` | oben links | Gemma-4-`box_2d` |

Dazu kommt: **PDF zählt y nativ von unten**, alle Bildwelten von oben. PyMuPDF
rechnet das bereits um — aber sobald Sie mit einer anderen Bibliothek arbeiten,
ist die Verwechslung eine vertikale Spiegelung.

> ⚠️ Deshalb ist im Schema unten der Bezugsrahmen **Pflichtfeld**, keine Konvention
> im Kopf. Und eine IoU-Berechnung über Rahmengrenzen hinweg wirft eine Exception,
> statt eine plausible Zahl zurückzugeben.

In [ ]:
class Ursprung(str, Enum):
    OBEN_LINKS = "oben_links"
    UNTEN_LINKS = "unten_links"


class Bezugsrahmen(str, Enum):
    MODELL_800 = "modell_800"
    BILD_PIXEL = "bild_pixel"
    SEITE_PUNKT = "seite_punkt"
    NORMIERT_1000 = "normiert_1000"


class Bbox(BaseModel):
    """Achsenparalleles Rechteck. Bezugsrahmen und Ursprung sind Pflicht."""
    x0: float
    y0: float
    x1: float
    y1: float
    rahmen: Bezugsrahmen
    ursprung: Ursprung = Ursprung.OBEN_LINKS

    @model_validator(mode="after")
    def _sortiert(self) -> "Bbox":
        if self.x1 < self.x0 or self.y1 < self.y0:
            raise ValueError(f"Bbox nicht sortiert: {self.x0},{self.y0} - {self.x1},{self.y1}")
        return self

    @property
    def breite(self) -> float:  return self.x1 - self.x0
    @property
    def hoehe(self) -> float:   return self.y1 - self.y0
    @property
    def flaeche(self) -> float: return self.breite * self.hoehe

    @classmethod
    def aus_cxcywh(cls, cx, cy, w, h, rahmen, **kw) -> "Bbox":
        """DETR gibt Mittelpunkt plus Größe aus, nicht Ecken."""
        return cls(x0=cx - w/2, y0=cy - h/2, x1=cx + w/2, y1=cy + h/2,
                   rahmen=rahmen, **kw)

    def iou(self, andere: "Bbox") -> float:
        if self.rahmen is not andere.rahmen or self.ursprung is not andere.ursprung:
            raise ValueError(
                f"IoU über Rahmengrenze: {self.rahmen.value}/{self.ursprung.value} "
                f"gegen {andere.rahmen.value}/{andere.ursprung.value}")
        x0, y0 = max(self.x0, andere.x0), max(self.y0, andere.y0)
        x1, y1 = min(self.x1, andere.x1), min(self.y1, andere.y1)
        if x1 <= x0 or y1 <= y0:
            return 0.0
        schnitt = (x1 - x0) * (y1 - y0)
        vereinigung = self.flaeche + andere.flaeche - schnitt
        return schnitt / vereinigung if vereinigung else 0.0

In [ ]:
def zurueck_ins_bild(box: Bbox, bild_breite: int, bild_hoehe: int) -> Bbox:
    """800x800 -> Bildpixel. Wegen keep_ratio=false eine reine achsenweise Streckung."""
    if box.rahmen is not Bezugsrahmen.MODELL_800:
        raise ValueError(f"Erwarte MODELL_800, bekommen {box.rahmen.value}")
    fx, fy = bild_breite / 800, bild_hoehe / 800
    return Bbox(x0=box.x0*fx, y0=box.y0*fy, x1=box.x1*fx, y1=box.y1*fy,
                rahmen=Bezugsrahmen.BILD_PIXEL, ursprung=box.ursprung)


def ins_pdf(box: Bbox, dpi: int) -> Bbox:
    """Bildpixel -> PDF-Punkte."""
    if box.rahmen is not Bezugsrahmen.BILD_PIXEL:
        raise ValueError(f"Erwarte BILD_PIXEL, bekommen {box.rahmen.value}")
    f = 72.0 / dpi
    return Bbox(x0=box.x0*f, y0=box.y0*f, x1=box.x1*f, y1=box.y1*f,
                rahmen=Bezugsrahmen.SEITE_PUNKT, ursprung=box.ursprung)


def nach_gemma(box: Bbox, bild_breite: int, bild_hoehe: int) -> list[int]:
    """Bildpixel -> Gemma-4-Format [y0, x0, y1, x1], normiert auf 1000x1000."""
    if box.rahmen is not Bezugsrahmen.BILD_PIXEL:
        raise ValueError(f"Erwarte BILD_PIXEL, bekommen {box.rahmen.value}")
    sx, sy = 1000/bild_breite, 1000/bild_hoehe
    return [round(box.y0*sy), round(box.x0*sx), round(box.y1*sy), round(box.x1*sx)]


# --- Rundlauf an der Zimbardo-Seite
bild, pg = seite_rendern(SEITEN["zimbardo"])
h, b = bild.shape[:2]

box_modell = Bbox.aus_cxcywh(400, 400, 100, 50, Bezugsrahmen.MODELL_800)
box_bild   = zurueck_ins_bild(box_modell, b, h)
box_pdf    = ins_pdf(box_bild, RENDER_DPI)

zeig = lambda bx: [round(v, 1) for v in (bx.x0, bx.y0, bx.x1, bx.y1)]
print("MODELL_800  ", zeig(box_modell))
print("BILD_PIXEL  ", zeig(box_bild))
print("SEITE_PUNKT ", zeig(box_pdf), f"  (Seite: {pg.rect.width:.0f} x {pg.rect.height:.0f} pt)")
print("Gemma box_2d", nach_gemma(box_bild, b, h))

# Der Wächter muss zuschlagen:
try:
    box_modell.iou(box_bild)
except ValueError as e:
    print("\nWächter greift:", e)

---
## 5. Die 25 Klassen und ihre Abbildung

Der Klassifikationskopf hat 25 Ausgänge. Der Index **ist** die Klassen-ID — aber
welcher Name zu welchem Index gehört, dazu gibt es zwei sich widersprechende Quellen:

| Quelle | Charakter |
|---|---|
| `label_list` in der PaddleX-`config.json` | 25 **verschiedene** Namen |
| `labels.json` / `id2label` des HF-Ports | 25 Einträge, aber nur **20 verschiedene** Namen |

Der HF-Port führt fünf Klassen mit ihren Nachbarn zusammen:

| ID | PaddleX | HF-Port |
|---|---|---|
| 5 | `display_formula` | `formula` |
| 9 | `footer_image` | `footer` |
| 13 | `header_image` | `header` |
| 15 | `inline_formula` | `formula` |
| 23 | `vertical_text` | `text` |

**Wir nehmen die PaddleX-Namen.** Begründung: die Gewichte sind dieselben, der Kopf
hat in beiden Fällen 25 getrennte Ausgänge, und die feineren Namen kosten nichts —
zusammenlegen kann man später immer, aufspalten nie. Die HF-Tabelle behalten wir als
Gegenprobe im Code, damit die Abweichung dokumentiert ist statt vergessen.

Als Zielformat verwenden wir `DoclingDocument` aus `docling-core`: Pydantic v2,
MIT-Lizenz, das etablierte Austauschformat der Werkzeuglandschaft. Die Abbildung
ist aber an mehreren Stellen **verlustbehaftet**.

> 💡 **Zentrale Entwurfsentscheidung:** Das native Label wird im Zwischenschema
> **wörtlich** mitgeführt. Die Abbildung auf `DocItemLabel` geschieht erst bei der
> Konsolidierung. Sonst wirft Stufe 1 Information weg, die Stufe 4 wieder braucht.

In [ ]:
# Reihenfolge exakt wie die label_list der PaddleX-config.json -- der Index IST die Klassen-ID.
PP_LABELS: list[str] = [
    "abstract", "algorithm", "aside_text", "chart", "content",
    "display_formula", "doc_title", "figure_title", "footer", "footer_image",
    "footnote", "formula_number", "header", "header_image", "image",
    "inline_formula", "number", "paragraph_title", "reference",
    "reference_content", "seal", "table", "text", "vertical_text",
    "vision_footnote",
]

# Gegenprobe: labels.json des ONNX-Exports. Gleiche Gewichte, gröbere Namen.
HF_ID2LABEL: dict[int, str] = {
    0: "abstract", 1: "algorithm", 2: "aside_text", 3: "chart", 4: "content",
    5: "formula", 6: "doc_title", 7: "figure_title", 8: "footer", 9: "footer",
    10: "footnote", 11: "formula_number", 12: "header", 13: "header", 14: "image",
    15: "formula", 16: "number", 17: "paragraph_title", 18: "reference",
    19: "reference_content", 20: "seal", 21: "table", 22: "text", 23: "text",
    24: "vision_footnote",
}

# (docling-Label, was beim Mappen verloren geht). None = verlustfrei.
PP_NACH_DOCLING: dict[str, tuple[str | None, str | None]] = {
    "doc_title":         ("title",          None),
    "paragraph_title":   ("section_header", None),
    "text":              ("text",           None),
    "figure_title":      ("caption",        None),
    "image":             ("picture",        None),
    "chart":             ("chart",          None),
    "table":             ("table",          None),
    "display_formula":   ("formula",        None),
    "footnote":          ("footnote",       None),
    "header":            ("page_header",    None),
    "footer":            ("page_footer",    None),
    "reference":         ("reference",      None),
    "content":           ("document_index", None),
    "aside_text":        ("text",      "Marginalie – eigener Lesestrom"),
    "inline_formula":    ("formula",   "inline statt abgesetzt"),
    "formula_number":    ("text",      "Formelnummer"),
    "abstract":          ("text",      "Rolle als Zusammenfassung"),
    "algorithm":         ("code",      "nur näherungsweise"),
    "reference_content": ("reference", "fällt mit reference zusammen"),
    "vision_footnote":   ("footnote",  "Bezug zur Abbildung"),
    "header_image":      ("picture",   "Zugehörigkeit zur Kopfzeile"),
    "footer_image":      ("picture",   "Zugehörigkeit zur Fußzeile"),
    "number":            ("text",      "Seitenzahl"),
    "vertical_text":     ("text",      "Schreibrichtung"),
    "seal":              ("picture",   "Siegeleigenschaft"),
}

# --- Selbsttest: vollständig und gegen das echte Enum geprüft?
from docling_core.types.doc import DocItemLabel
gueltig = {l.value for l in DocItemLabel}

assert len(PP_LABELS) == len(HF_ID2LABEL) == 25, "Der Kopf hat 25 Ausgänge."
fehlend = [l for l in PP_LABELS if l not in PP_NACH_DOCLING]
ungueltig = {k: v for k, (v, _) in PP_NACH_DOCLING.items() if v and v not in gueltig}
verlustfrei = [l for l in PP_LABELS if PP_NACH_DOCLING[l][1] is None]

assert not fehlend,   f"ohne Mapping: {fehlend}"
assert not ungueltig, f"kein gültiges DocItemLabel: {ungueltig}"

abweichung = [(i, PP_LABELS[i], HF_ID2LABEL[i])
              for i in range(25) if PP_LABELS[i] != HF_ID2LABEL[i]]

print(f"{len(PP_LABELS)} Klassen, {len(verlustfrei)} verlustfrei, "
      f"{len(PP_LABELS) - len(verlustfrei)} verlustbehaftet")
print(f"{len(abweichung)} Abweichungen gegenüber labels.json des Exports: "
      + ", ".join(f"{i}:{a}/{b}" for i, a, b in abweichung) + "\n")

for i, l in enumerate(PP_LABELS):
    ziel, verlust = PP_NACH_DOCLING[l]
    marke = "  " if verlust is None else "! "
    print(f"{marke}{i:2d} {l:19s} -> {str(ziel):16s} {verlust or ''}")

### Der wichtigste Fund: `aside_text`

Eine offene Frage des Projekts war, wie sich **Randnotizen vom Haupttext trennen**
lassen — die grün hinterlegte Marginalspalte der Wahrnehmungspsychologie-Seite
enthält Text, gehört aber nicht in den Lesefluss.

Antwort: **Der Detektor macht das selbst.** `aside_text` ist genau diese Spalte,
und sie hat in beiden Label-Tabellen denselben Namen — die Abweichung von oben
betrifft sie nicht.

`docling` kennt dafür kein Label — ein weiterer Grund, das native Label zu behalten
und daraus einen eigenen **Strom**-Marker abzuleiten.

In [ ]:
class Strom(str, Enum):
    HAUPT = "haupt"
    MARGINALIE = "marginalie"
    BOILERPLATE = "boilerplate"
    APPARAT = "apparat"          # Fußnoten, Referenzen

BOILERPLATE = {"header", "footer", "header_image", "footer_image", "number"}
APPARAT = {"footnote", "vision_footnote", "reference", "reference_content"}

def strom_fuer(pp_label: str) -> Strom:
    if pp_label in BOILERPLATE:  return Strom.BOILERPLATE
    if pp_label == "aside_text": return Strom.MARGINALIE
    if pp_label in APPARAT:      return Strom.APPARAT
    return Strom.HAUPT

for l in ["text", "aside_text", "footer", "footnote", "display_formula"]:
    print(f"{l:16s} -> {strom_fuer(l).value}")

> 💡 **Nebeneffekt für den Datenschutz:** Der Strom `BOILERPLATE` fängt genau die
> Elemente, die man vor der Weiterverarbeitung entfernen will. Auf der
> Zimbardo-Seite steht in der Fußzeile ein Personalisierungsstempel
> („Persönliches Exemplar von …") — Boilerplate **und** personenbezogenes Datum.

---
## 6. Das Zwischenschema

Bewusst **nicht** direkt `DoclingDocument`. Der Grund ist die Aufgabe: In der
Detektionsphase müssen konkurrierende Kandidaten mit Quelle und Konfidenz gehalten
werden. `DoclingDocument` ist ein Ergebnismodell und kann das nicht.

```
SeitenBefund
 ├── Metadaten (Datei, Seite, Maße, dpi)
 ├── Block[]      id, query_id, pp_label, score, Bbox, Polygon, lese_index, Quelle
 ├── Lesekante[]  von -> nach mit Konfidenz
 └── Warnungen[]
```

### Warum Rang **und** Kanten

Die Referenz-Nachverarbeitung löst die 300×300-Zeigermatrix zu einem
**ganzzahligen Rang pro Block** auf. Dieser Rang ist die belastbare Größe — wir
führen ihn als `lese_index`.

Die Kanten bleiben trotzdem im Schema, aus zwei Gründen. Erstens lässt sich für
jedes benachbarte Paar der Endreihenfolge die Wahrscheinlichkeit direkt aus der
Matrix ablesen — das ist ein echtes Gütemaß und keine erfundene Zahl. Zweitens
liefern andere Exporte desselben Modells *nur* einen Rang und keine Matrix; dann
bleibt `kanten` eben leer, und `lesefolge()` funktioniert weiter.

`id` ist ein laufender Index innerhalb des Befunds, `query_id` der Index der
Detektor-Query. Beide getrennt zu halten ist nötig, weil die Auswahl per Top-k über
das *(Query × Klasse)*-Gitter läuft: dieselbe Query kann mit zwei verschiedenen
Klassen zweimal über der Schwelle landen.

In [ ]:
class Block(BaseModel):
    """Ein detektiertes Layout-Element."""
    id: int = Field(description="laufender Index innerhalb des Befunds")
    query_id: int = Field(description="Index der Query im Detektor, 0..299")
    pp_label: str = Field(description="Native Klasse aus der label_list – wörtlich")
    score: float = Field(ge=0.0, le=1.0)
    bbox: Bbox
    lese_index: int | None = Field(
        default=None, description="Rang aus der Zeigermatrix; kleiner = früher")
    polygon: list[tuple[float, float]] | None = Field(
        default=None, description="aus der Instanzmaske abgeleitet, falls vorhanden")
    quelle: Literal["pp_doclayout", "pymupdf", "vlm", "mensch"] = "pp_doclayout"
    text: str | None = Field(default=None, description="erst ab Stufe 2/3 gefüllt")

    @property
    def docling_label(self) -> str | None:
        return PP_NACH_DOCLING.get(self.pp_label, (None, None))[0]

    @property
    def verlust(self) -> str | None:
        """Was beim Mappen nach docling verloren geht. None = verlustfrei."""
        return PP_NACH_DOCLING.get(self.pp_label, (None, None))[1]

    @property
    def strom(self) -> Strom:
        return strom_fuer(self.pp_label)


class Lesekante(BaseModel):
    """Gerichtete Kante aus der 300x300-Zeigermatrix: von -> nach.

    `von` und `nach` sind Block-`id`, nicht `query_id`.
    """
    von: int
    nach: int
    konfidenz: float = Field(ge=0.0, le=1.0)
    marge: float | None = Field(
        default=None,
        description="Logit-Marge für 'nach folgt auf von'. Sättigt nicht, "
                    "anders als konfidenz. Positiv = Modell stimmt zu.")


class SeitenBefund(BaseModel):
    """Umschlag: alles, was Stufe 1 über eine Seite weiß."""
    quelle_datei: str
    seite: int = Field(ge=0)
    seite_breite_pt: float
    seite_hoehe_pt: float
    render_dpi: int
    bild_breite_px: int
    bild_hoehe_px: int
    bloecke: list[Block] = Field(default_factory=list)
    kanten: list[Lesekante] = Field(default_factory=list)
    warnungen: list[str] = Field(default_factory=list)

    def lesefolge(self, strom: Strom = Strom.HAUPT) -> list[Block]:
        """Blöcke eines Stroms in Lesereihenfolge.

        Bevorzugt den `lese_index` aus der Zeigermatrix. Fehlt er, wird greedy über
        die Kantenkonfidenz gelaufen: Startpunkt ist der Block ohne eingehende
        Kante, danach jeweils die stärkste ausgehende Kante.
        """
        erlaubt = {b.id: b for b in self.bloecke if b.strom is strom}
        if not erlaubt:
            return []

        if all(b.lese_index is not None for b in erlaubt.values()):
            return sorted(erlaubt.values(), key=lambda b: (b.lese_index, b.id))

        kanten = [k for k in self.kanten if k.von in erlaubt and k.nach in erlaubt]
        ziele = {k.nach for k in kanten}
        aktuell = next((i for i in erlaubt if i not in ziele), min(erlaubt))
        folge, gesehen = [], set()
        while aktuell is not None and aktuell not in gesehen:
            gesehen.add(aktuell)
            folge.append(erlaubt[aktuell])
            weiter = [k for k in kanten if k.von == aktuell and k.nach not in gesehen]
            aktuell = max(weiter, key=lambda k: k.konfidenz).nach if weiter else None
        folge += [b for i, b in erlaubt.items() if i not in gesehen]
        return folge

    def schwaechste_kanten(self, anzahl: int = 5) -> list[Lesekante]:
        """Die unsichersten Übergänge - der billigste Einstieg in die Fehlersuche.

        Sortiert nach der Logit-Marge, weil `konfidenz` bei sicheren Seiten
        durchgehend auf 1.0 sättigt und dann nicht mehr unterscheidet.
        """
        if self.kanten and all(k.marge is not None for k in self.kanten):
            return sorted(self.kanten, key=lambda k: k.marge)[:anzahl]
        return sorted(self.kanten, key=lambda k: k.konfidenz)[:anzahl]

In [ ]:
# --- Trockentest mit erfundenen Blöcken (der Detektor läuft erst in Abschnitt 7)
bild, pg = seite_rendern(SEITEN["wahrnehmung"])
h, b = bild.shape[:2]

def _bx(x0, y0, x1, y1):
    return Bbox(x0=x0, y0=y0, x1=x1, y1=y1, rahmen=Bezugsrahmen.BILD_PIXEL)

befund_test = SeitenBefund(
    quelle_datei=SEITEN["wahrnehmung"].name, seite=0,
    seite_breite_pt=pg.rect.width, seite_hoehe_pt=pg.rect.height,
    render_dpi=RENDER_DPI, bild_breite_px=b, bild_hoehe_px=h,
    bloecke=[
        Block(id=0, query_id=17,  pp_label="header",     score=0.97, lese_index=0, bbox=_bx(120, 80, 570, 110)),
        Block(id=1, query_id=44,  pp_label="text",       score=0.94, lese_index=2, bbox=_bx(900,180,1410, 320)),
        Block(id=2, query_id=91,  pp_label="aside_text", score=0.91, lese_index=1, bbox=_bx(130,185, 450, 300)),
        Block(id=3, query_id=112, pp_label="image",      score=0.96, lese_index=3, bbox=_bx(510,185, 865, 610)),
        Block(id=4, query_id=203, pp_label="text",       score=0.93, lese_index=4, bbox=_bx(510,650,1410, 850)),
    ],
    kanten=[Lesekante(von=1, nach=3, konfidenz=0.88),
            Lesekante(von=3, nach=4, konfidenz=0.61)],
)

print("Hauptstrom in Lesereihenfolge:")
for blk in befund_test.lesefolge():
    print(f"  #{blk.id} lese_index={blk.lese_index}  {blk.pp_label:12s} -> {blk.docling_label}")
print("\nMarginalien (eigener Strom):")
for blk in befund_test.lesefolge(Strom.MARGINALIE):
    print(f"  #{blk.id} {blk.pp_label:12s} Verlust beim Mappen: {blk.verlust}")

print("\nJSON-Ausschnitt:")
print(json.dumps(befund_test.model_dump(mode="json")["bloecke"][2], indent=2, ensure_ascii=False))

---
## 7. Der Detektor — Selbsttest zuerst

Ab hier wird die ONNX-Datei gebraucht. **Bevor** Sie inferieren: nachsehen, welchen
Vertrag der Export tatsächlich hat. Verschiedene Exporte desselben Modells
unterscheiden sich erheblich, und keiner sagt es Ihnen zur Laufzeit.

Zwei praktische Fallen:

1. **Provider.** `onnxruntime` wirft, wenn ein nicht vorhandener Execution Provider
   angefordert wird. Auf dem Mac gibt es kein CUDA. Deshalb wird die Wunschliste
   gegen `get_available_providers()` gefiltert, statt sie hart zu setzen.
2. **Die falsche Bibliothek.** `onnx.load()` gehört ins Paket `onnx`, nicht in
   `onnxruntime`. Wir lesen die Signatur stattdessen aus der Session — dieselbe
   Information, eine Abhängigkeit weniger.

> 💡 **Zu CoreML auf Apple Silicon:** Der CoreML-Provider fällt bei DETR-Graphen
> häufig teilweise auf CPU zurück und kostet dann mehr, als er bringt. Der Default
> unten ist deshalb reines CPU. Messen Sie mit `nur_cpu=False` nach, bevor Sie
> umschalten.

In [ ]:
def provider_kette(nur_cpu: bool = True) -> list[str]:
    """Nur real verfügbare Provider anfordern - sonst wirft ORT auf dem Mac."""
    if nur_cpu:
        return ["CPUExecutionProvider"]
    verfuegbar = set(ort.get_available_providers())
    wunsch = ["CUDAExecutionProvider", "CoreMLExecutionProvider", "CPUExecutionProvider"]
    return [p for p in wunsch if p in verfuegbar] or ["CPUExecutionProvider"]


def sitzung_oeffnen(pfad: Path, nur_cpu: bool = True,
                    threads: int | None = None) -> ort.InferenceSession:
    opt = ort.SessionOptions()
    opt.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    if threads:
        opt.intra_op_num_threads = threads
    return ort.InferenceSession(str(pfad), sess_options=opt,
                                providers=provider_kette(nur_cpu))


def signatur_zeigen(sitzung: ort.InferenceSession) -> dict[str, list]:
    """Pflicht-Selbsttest: welchen Vertrag hat dieser Export wirklich?"""
    print("Provider:", sitzung.get_providers())
    print("\nEingänge:")
    for e in sitzung.get_inputs():
        print(f"  {e.name:16s} {e.type:22s} {e.shape}")
    print("Ausgänge:")
    for a in sitzung.get_outputs():
        print(f"  {a.name:16s} {a.type:22s} {a.shape}")

    namen = [a.name for a in sitzung.get_outputs()]
    erwartet = ["logits", "pred_boxes", "order_logits"]
    fehlend = [k for k in erwartet if k not in namen]
    if fehlend:
        raise RuntimeError(
            f"Export unvollständig, es fehlen: {fehlend}. "
            "Dieses Notebook braucht die Rohköpfe (phungpx/PP-DocLayoutV3-ONNX). "
            "Exporte mit eingebackenem Postprocess liefern stattdessen einen "
            "einzelnen Tensor (300, 7).")
    if "out_masks" not in namen:
        print("\n! 'out_masks' fehlt - Polygone fallen auf die Bounding Box zurück.")
    else:
        print("\nAlle vier Köpfe vorhanden.")
    return {"eingaenge": [e.name for e in sitzung.get_inputs()], "ausgaenge": namen}


SITZUNG = sitzung_oeffnen(ONNX_DATEI)
SIGNATUR = signatur_zeigen(SITZUNG)
EINGANG = SIGNATUR["eingaenge"][0]

### Die Nachverarbeitung

Vier Schritte, alle 1:1 aus der Referenz-Implementierung `pp_doclayout_v3_onnx.py`
portiert, die ihrerseits aus `PPDocLayoutV3ImageProcessor` stammt:

1. **Auswahl.** `sigmoid(logits)` ergibt ein 300×25-Gitter. Top-k über das *flach
   gemachte* Gitter mit k = 300, dann Schwelle. Kein NMS. Weil über *(Query, Klasse)*
   ausgewählt wird, kann dieselbe Query zweimal auftauchen — deshalb `query_id`
   getrennt von `id`.
2. **Boxen.** `cxcywh` normiert → `xyxy`, multipliziert mit der Größe des
   **Originalbildes**. Die Boxen landen also direkt in `BILD_PIXEL`; der Umweg über
   `MODELL_800` entfällt in der Praxis und bleibt nur als didaktischer Rundlauf in
   Abschnitt 4.
3. **Lesereihenfolge.** Die Zeigermatrix wird sigmoidiert; für jede Query zählt
   `votes[j] = Σ_{i<j} s[i,j] + Σ_{i>j} (1 − s[j,i])`. Aufsteigend sortiert ergibt
   das den Rang. Für ein benachbartes Paar (a, b) der Endreihenfolge ist
   `s[a,b]` bzw. `1 − s[b,a]` die Wahrscheinlichkeit, dass b auf a folgt — genau die
   Kantenkonfidenz.
4. **Polygone.** Maskenlogits > Schwelle, Ausschnitt in Maskenkoordinaten,
   auf Boxgröße hochskalieren, größte Kontur, `approxPolyDP`. Fällt bei leerer Maske
   auf das Rechteck zurück.

In [ ]:
MASKEN_STRIDE = 4        # Maskenkopf hat 800/4 = 200 Kantenlänge


def _sigmoid(x: np.ndarray) -> np.ndarray:
    return (1.0 / (1.0 + np.exp(-x.astype(np.float64)))).astype(np.float32)


def lese_raenge(order_logits: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """(Q, Q) Zeigerlogits -> (Rang je Query, sigmoidierte Matrix).

    Portiert aus `get_order_seqs`. Die Matrix wird mit zurückgegeben, weil sie die
    paarweisen Konfidenzen trägt.
    """
    s = _sigmoid(order_logits)
    q = s.shape[0]
    stimmen = np.triu(s, 1).sum(axis=0) + np.tril(1.0 - s.T, -1).sum(axis=0)
    zeiger = np.argsort(stimmen, kind="stable")
    rang = np.empty(q, dtype=np.int64)
    rang[zeiger] = np.arange(q)
    return rang, s


def folgt_auf(s: np.ndarray, a: int, b: int) -> float:
    """Wahrscheinlichkeit, dass Query b auf Query a folgt - aus der Zeigermatrix."""
    return float(s[a, b]) if a < b else float(1.0 - s[b, a])

def marge_fuer(order_logits: np.ndarray, a: int, b: int) -> float:
    """Vorzeichenbehaftete Logit-Marge für 'Query b folgt auf Query a'.

    Dieselbe Aussage wie folgt_auf(), aber vor der Sigmoid: sigmoid(marge)
    ergibt die Konfidenz zurück. Weil float32 jenseits von etwa +-17 auf
    1.0 bzw. 0.0 sättigt, ist nur die Marge als Gütemaß brauchbar.
    """
    return float(order_logits[a, b]) if a < b else float(-order_logits[b, a])

def _ecken_filtern(polygon: np.ndarray, spitzer_winkel: float = 45.0) -> list[tuple]:
    """Portiert aus `_extract_custom_vertices` der Referenz-Implementierung.

    Behält nur Ecken mit negativem Kreuzprodukt und rückt Ecken mit ~45° nach außen.
    Wirkt gegen die Treppenstufen, die aus der 200x200-Maske stammen.
    """
    poly = np.asarray(polygon, dtype=np.float64)
    n = len(poly)
    res = []
    for i in range(n):
        vorher, hier, danach = poly[(i - 1) % n], poly[i], poly[(i + 1) % n]
        v1, v2 = vorher - hier, danach - hier
        if (v1[1] * v2[0]) - (v1[0] * v2[1]) >= 0:
            continue
        n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
        if n1 == 0 or n2 == 0:
            res.append(tuple(hier))
            continue
        winkel = np.degrees(np.arccos(np.clip((v1 @ v2) / (n1 * n2), -1.0, 1.0)))
        if abs(winkel - spitzer_winkel) < 1:
            richtung = v1 / n1 + v2 / n2
            richtung = richtung / np.linalg.norm(richtung)
            res.append(tuple(hier + richtung * ((n1 + n2) / 2)))
        else:
            res.append(tuple(hier))
    return res


def _maske_zu_polygon(maske: np.ndarray, epsilon_anteil: float = 0.004):
    konturen, _ = cv2.findContours(maske, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not konturen:
        return None
    kontur = max(konturen, key=cv2.contourArea)
    eps = epsilon_anteil * cv2.arcLength(kontur, True)
    punkte = np.atleast_2d(cv2.approxPolyDP(kontur, eps, True).squeeze())
    if punkte.ndim != 2 or len(punkte) < 4:
        return None
    gefiltert = _ecken_filtern(punkte)
    return gefiltert if len(gefiltert) >= 4 else None


def polygone_ziehen(boxen: np.ndarray, masken: np.ndarray,
                    bild_breite: int, bild_hoehe: int) -> list[list[tuple[float, float]]]:
    """Instanzmasken -> Polygone in Bildpixeln. Fällt auf das Rechteck zurück."""
    sx = (800 / bild_breite) / MASKEN_STRIDE
    sy = (800 / bild_hoehe) / MASKEN_STRIDE
    mh, mw = masken.shape[1:]
    ergebnis = []

    for i in range(len(boxen)):
        x0, y0, x1, y1 = boxen[i].astype(np.int32)
        bw, bh = int(x1 - x0), int(y1 - y0)
        rechteck = [(float(x0), float(y0)), (float(x1), float(y0)),
                    (float(x1), float(y1)), (float(x0), float(y1))]
        if bw <= 0 or bh <= 0:
            ergebnis.append(rechteck)
            continue

        xa, xe = np.clip([int(round(x0 * sx)), int(round(x1 * sx))], 0, mw)
        ya, ye = np.clip([int(round(y0 * sy)), int(round(y1 * sy))], 0, mh)
        ausschnitt = masken[i, ya:ye, xa:xe]
        if ausschnitt.size == 0 or ausschnitt.sum() == 0:
            ergebnis.append(rechteck)
            continue

        gross = cv2.resize(ausschnitt.astype(np.uint8), (bw, bh),
                           interpolation=cv2.INTER_NEAREST)
        poly = _maske_zu_polygon(gross)
        if poly is None:
            ergebnis.append(rechteck)
            continue
        ergebnis.append([(float(px + x0), float(py + y0)) for px, py in poly])
    return ergebnis


def dekodieren(ausgaben: dict[str, np.ndarray], bild_breite: int, bild_hoehe: int,
               schwelle: float = 0.5) -> tuple[list[Block], list[Lesekante]]:
    """Rohtensoren -> Blöcke (nach Lesereihenfolge sortiert) und Lesekanten."""
    logits = ausgaben["logits"][0]                 # (Q, C)
    pred_boxes = ausgaben["pred_boxes"][0]         # (Q, 4) cxcywh normiert
    order_logits = ausgaben["order_logits"][0]     # (Q, Q)
    out_masks = ausgaben.get("out_masks")

    anzahl_queries, anzahl_klassen = logits.shape

    # 1. Auswahl: Top-k über das flache (Query x Klasse)-Gitter, dann Schwelle
    flach = _sigmoid(logits).reshape(-1)
    top = np.argpartition(-flach, anzahl_queries - 1)[:anzahl_queries]
    top = top[np.argsort(-flach[top], kind="stable")]
    scores = flach[top]
    klassen = top % anzahl_klassen
    queries = top // anzahl_klassen

    behalten = scores >= schwelle
    scores, klassen, queries = scores[behalten], klassen[behalten], queries[behalten]

    # 2. Boxen: cxcywh normiert -> xyxy in Bildpixeln
    mitte, groesse = pred_boxes[..., :2], pred_boxes[..., 2:]
    xyxy = np.concatenate([mitte - 0.5 * groesse, mitte + 0.5 * groesse], axis=-1)
    xyxy = xyxy * np.array([bild_breite, bild_hoehe, bild_breite, bild_hoehe], dtype=np.float32)
    boxen = xyxy[queries]

    # 3. Lesereihenfolge
    rang, matrix = lese_raenge(order_logits)
    ordnung = rang[queries]
    sortiert = np.argsort(ordnung, kind="stable")
    scores, klassen, queries = scores[sortiert], klassen[sortiert], queries[sortiert]
    boxen, ordnung = boxen[sortiert], ordnung[sortiert]

    # 4. Polygone
    if out_masks is not None and len(boxen):
        masken = (_sigmoid(out_masks[0][queries]) > schwelle).astype(np.uint8)
        polygone = polygone_ziehen(boxen, masken, bild_breite, bild_hoehe)
    else:
        polygone = [None] * len(boxen)

    bloecke = [
        Block(
            id=i,
            query_id=int(queries[i]),
            pp_label=PP_LABELS[int(klassen[i])],
            score=float(scores[i]),
            lese_index=int(ordnung[i]),
            bbox=Bbox(x0=float(boxen[i][0]), y0=float(boxen[i][1]),
                      x1=float(boxen[i][2]), y1=float(boxen[i][3]),
                      rahmen=Bezugsrahmen.BILD_PIXEL),
            polygon=polygone[i],
        )
        for i in range(len(boxen))
    ]

    # Kanten zwischen benachbarten Blöcken der Endreihenfolge. Trägt dieselbe Query
    # zwei Labels, gibt es keinen Übergang zwischen ihnen - diese Paare werden
    # übersprungen, statt eine bedeutungslose Zahl zu erzeugen.
    kanten = [
        Lesekante(von=i, nach=i + 1,
                  konfidenz=folgt_auf(matrix, int(queries[i]), int(queries[i + 1])),
                  marge=marge_fuer(order_logits, int(queries[i]), int(queries[i + 1])))
        for i in range(len(bloecke) - 1)
        if int(queries[i]) != int(queries[i + 1])
    ]
    return bloecke, kanten

In [ ]:
def layout_erkennen(pfad: Path, dpi: int = RENDER_DPI, seite: int = 0,
                    schwelle: float = 0.5, sitzung: ort.InferenceSession | None = None
                    ) -> SeitenBefund:
    """Stufe 1 komplett: PDF-Seite -> SeitenBefund."""
    sitzung = sitzung or SITZUNG
    bild, pg = seite_rendern(pfad, dpi, seite)
    h, b = bild.shape[:2]
    tensor = vorverarbeiten(bild)

    namen = [a.name for a in sitzung.get_outputs()]
    roh = sitzung.run(None, {EINGANG: tensor})
    ausgaben = dict(zip(namen, roh))

    bloecke, kanten = dekodieren(ausgaben, b, h, schwelle)

    warnungen = []
    if not bloecke:
        warnungen.append(f"Keine Detektion über der Schwelle {schwelle}.")
    if "out_masks" not in ausgaben:
        warnungen.append("Export ohne Maskenkopf - Polygone sind Rechtecke.")

    return SeitenBefund(
        quelle_datei=pfad.name, seite=seite,
        seite_breite_pt=pg.rect.width, seite_hoehe_pt=pg.rect.height,
        render_dpi=dpi, bild_breite_px=b, bild_hoehe_px=h,
        bloecke=bloecke, kanten=kanten, warnungen=warnungen)


BEFUNDE: dict[str, SeitenBefund] = {}
for name, pfad in SEITEN.items():
    if not pfad.exists():
        continue
    t0 = time.perf_counter()
    BEFUNDE[name] = layout_erkennen(pfad)
    dauer = time.perf_counter() - t0
    bf = BEFUNDE[name]
    verteilung = {}
    for blk in bf.bloecke:
        verteilung[blk.strom.value] = verteilung.get(blk.strom.value, 0) + 1
    print(f"{name:12s} {len(bf.bloecke):3d} Blöcke, {len(bf.kanten):3d} Kanten, "
          f"{dauer:5.2f} s   {verteilung}")
    for w in bf.warnungen:
        print(f"             ! {w}")

In [ ]:
# --- Was steht drin? Hauptstrom der Wahrnehmungspsychologie-Seite
bf = BEFUNDE["wahrnehmung"]

print("Hauptstrom in Lesereihenfolge:")
for blk in bf.lesefolge():
    ecken = len(blk.polygon) if blk.polygon else 0
    print(f"  {blk.lese_index:3d}  {blk.pp_label:18s} {blk.score:.2f}  "
          f"{ecken:2d} Polygonecken  -> {blk.docling_label}")

print("\nMarginalien:")
for blk in bf.lesefolge(Strom.MARGINALIE):
    print(f"  {blk.lese_index:3d}  {blk.pp_label:18s} {blk.score:.2f}")

print("\nBoilerplate (Kandidaten zum Verwerfen):")
for blk in bf.lesefolge(Strom.BOILERPLATE):
    print(f"  {blk.lese_index:3d}  {blk.pp_label:18s} {blk.score:.2f}")

print("\nUnsicherste Übergänge:")
nach_id = {b.id: b for b in bf.bloecke}
for k in bf.schwaechste_kanten():
    a, z = nach_id[k.von], nach_id[k.nach]
    print(f"  {a.pp_label:18s} -> {z.pp_label:18s}  "
          f"marge {k.marge:+8.2f}   konfidenz {k.konfidenz:.3f}")

### `is_scale` nachmessen statt glauben

Die Vorverarbeitung teilt durch 255. Das steht so in der Referenz-Implementierung,
folgt aber aus keiner der beiden Konfigurationsdateien zwingend. Die folgende Zelle
lässt beide Varianten laufen und vergleicht, was herauskommt.

Erwartung: mit 0…255 bricht die Detektion weitgehend zusammen, weil das Netz
Aktivierungen weit außerhalb des Trainingsbereichs sieht. Falls beide Varianten
ähnlich viele Blöcke liefern, ist die Annahme falsch — dann bitte melden.

In [ ]:
bild_p, _ = seite_rendern(SEITEN["zimbardo"])
h_p, b_p = bild_p.shape[:2]
namen_p = [a.name for a in SITZUNG.get_outputs()]

for beschriftung, skalieren in [("0..1  (is_scale=True)", True), ("0..255 (is_scale=False)", False)]:
    t = vorverarbeiten(bild_p, skaliere_auf_eins=skalieren)
    aus = dict(zip(namen_p, SITZUNG.run(None, {EINGANG: t})))
    bl, _ = dekodieren(aus, b_p, h_p)
    hoechst = float(_sigmoid(aus["logits"][0]).max())
    print(f"{beschriftung:26s} {len(bl):3d} Blöcke über 0.5   "
          f"höchster Score {hoechst:.3f}")

---
## 8. Kontrollbild

Zahlen lügen leiser als Bilder. Jede Layout-Ausgabe gehört einmal aufs Blatt gelegt,
bevor man ihr traut — das ist die billigste Qualitätssicherung der ganzen Pipeline.

Gezeichnet werden Polygone (nicht nur Rechtecke), der Rang aus der Lesereihenfolge
und die Übergänge. Die Farbe kodiert den **Strom**, nicht die Klasse: so sieht man
auf einen Blick, ob die Marginalspalte sauber aus dem Hauptstrom herausgehalten wurde.

In [ ]:
# Farben als RGB notiert - die Umrechnung nach BGR passiert einmal, in der Funktion.
FARBEN_RGB = {
    Strom.HAUPT:       (9, 105, 218),     # Hausfarbe #0969DA
    Strom.MARGINALIE:  (23, 138, 63),
    Strom.BOILERPLATE: (130, 130, 130),
    Strom.APPARAT:     (191, 121, 15),
}
def _bgr(rgb): return (rgb[2], rgb[1], rgb[0])


def overlay(befund: SeitenBefund, bild: np.ndarray, mit_reihenfolge: bool = True,
            mit_polygon: bool = True) -> np.ndarray:
    leinwand = bild.copy()[:, :, ::-1].copy()          # RGB -> BGR für cv2
    mitte = {}
    for blk in befund.bloecke:
        bx = blk.bbox
        farbe = _bgr(FARBEN_RGB[blk.strom])
        if mit_polygon and blk.polygon and len(blk.polygon) >= 4:
            poly = np.array(blk.polygon, dtype=np.int32).reshape(-1, 1, 2)
            cv2.polylines(leinwand, [poly], True, farbe, 2)
        else:
            cv2.rectangle(leinwand, (int(bx.x0), int(bx.y0)),
                          (int(bx.x1), int(bx.y1)), farbe, 2)
        cv2.putText(leinwand, f"{blk.lese_index}:{blk.pp_label} {blk.score:.2f}",
                    (int(bx.x0) + 3, max(14, int(bx.y0) - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, farbe, 1, cv2.LINE_AA)
        mitte[blk.id] = (int((bx.x0 + bx.x1) / 2), int((bx.y0 + bx.y1) / 2))

    if mit_reihenfolge:
        for k in befund.kanten:
            if k.von in mitte and k.nach in mitte:
                # unsichere Übergänge kräftiger rot, sichere blasser
                rot = int(60 + 195 * (1.0 - k.konfidenz))
                cv2.arrowedLine(leinwand, mitte[k.von], mitte[k.nach],
                                _bgr((rot, 20, 20)), 2, tipLength=0.02)
    return leinwand[:, :, ::-1]                        # zurück nach RGB


from IPython.display import Image, display

AUSGABE = Path("kontrolle")
AUSGABE.mkdir(exist_ok=True)

for name in BEFUNDE:
    bild_k, _ = seite_rendern(SEITEN[name])
    kontrolle = overlay(BEFUNDE[name], bild_k)
    ziel = AUSGABE / f"kontrolle_{name}.png"
    cv2.imwrite(str(ziel), kontrolle[:, :, ::-1])
    print(f"{ziel}  {kontrolle.shape}")

display(Image(str(AUSGABE / "kontrolle_wahrnehmung.png"), width=760))

In [ ]:
# --- Befunde als JSON ablegen: die Schnittstelle zu Stufe 2
BEFUND_DIR = Path("befunde")
BEFUND_DIR.mkdir(exist_ok=True)

for name, bf in BEFUNDE.items():
    ziel = BEFUND_DIR / f"{name}_stufe1.json"
    ziel.write_text(bf.model_dump_json(indent=2), encoding="utf-8")
    print(f"{ziel}  {ziel.stat().st_size / 1024:.1f} kB")

---
## 9. Offene Punkte

Ehrlichkeit über das, was noch **nicht** geklärt ist, gehört zur Dokumentation:

| # | Punkt | Wie zu klären |
|---|---|---|
| 1 | Zwei widersprüchliche `id2label`-Tabellen | Handannotation einer Seite mit einer Formel im Fließtext und einer abgesetzten: trennt der Kopf 5 und 15 wirklich? |
| 2 | Kantenkonfidenz als Gütemaß | ist `folgt_auf()` kalibriert? Gegen das Goldset aus Aufgabe E prüfen |
| 3 | Doppelte Queries | dieselbe Query kann mit zwei Klassen über der Schwelle liegen. Zusammenlegen oder als konkurrierende Kandidaten behalten? |
| 4 | Polygonqualität | `approxPolyDP` mit `epsilon_ratio=0.004` ist der Referenzwert. Für mehrspaltig umflossene Abbildungen evtl. zu grob |
| 5 | Kästen ohne Klasse | „Für die Praxis", „Definition" haben in den 25 Klassen **kein** Gegenstück |
| 6 | Goldset | ohne Handannotation ist „robust" Geschmackssache |

Zu Punkt 3: Der Referenzcode legt **nicht** zusammen, er lässt Doppelungen stehen.
Das ist für uns eher ein Vorteil — konkurrierende Kandidaten mit Konfidenz sind
genau das, wofür das Zwischenschema gebaut wurde. Nur muss Stufe 4 sie auflösen.

Zu Punkt 5: Das ist kein Modellfehler, sondern die Lücke, die Stufe 4 füllt. Dort
typisiert ein Vision-LLM den Kasten — und dort helfen die **Vektorobjekte** aus dem
Seiteninventar als deterministischer Regionsnachweis. Auf der
Wahrnehmungspsychologie-Seite ist die Marginalspalte ein einzelnes gefülltes
Rechteck, der Praxiskasten ein Rahmen plus Kopfbalken in der Hausfarbe des Verlags.
Exakte Regionsgrenzen, ohne jedes Modell.

---
## 10. Übungsaufgaben

**A — Koordinaten (Einstieg).**
Eine Box liegt bei `cxcywh = (600, 200, 120, 60)` im Modellraum. Die Seite ist
595 × 793 pt, gerendert bei 150 dpi. Rechnen Sie die Box von Hand in PDF-Punkte um
und prüfen Sie das Ergebnis mit den Funktionen aus Abschnitt 4. Warum sind die
Streckungsfaktoren für x und y verschieden?

**B — Verlust benennen (mittel).**
Suchen Sie in der Tabelle aus Abschnitt 5 die drei Abbildungen heraus, bei denen der
Verlust *fachlich* am schwersten wiegt, und begründen Sie es an einem konkreten
Dokumenttyp.

**C — Triage härten (mittel).**
Die Regel „`get_text()` leer ⇒ Scan" scheitert an der Tietze-Schenk-Seite. Entwerfen
Sie eine bessere Regel, die den Textlayer *validiert*, statt nur seine Existenz zu
prüfen. Tipp: Vergleichen Sie die Schriftnamen mit einer Liste typischer
Substitutionsschriften, und messen Sie den Anteil ungewöhnlicher Zeichenfolgen.

**D — Ströme erweitern (fortgeschritten).**
`strom_fuer()` bildet vier Ströme ab. Auf der Wahrnehmungspsychologie-Seite gibt es
einen fünften Fall: den Kasten „Für die Praxis" mit **zwei inneren Spalten**.
Erweitern Sie das Schema um Container mit Kindern. Welche Felder brauchen Sie
mindestens, und was passiert mit der Lesereihenfolge?

**E — Goldset (Projektaufgabe).**
Annotieren Sie eine der drei Seiten von Hand: Label, Bbox, Lesereihenfolge, als YAML
im Repository. Berechnen Sie anschließend für den Detektorlauf die
Kantengenauigkeit der Lesereihenfolge. Zum Vergleich: Auf dem mehrspaltigen Teil
von OmniDocBench erreicht der klassische rekursive XY-Cut rund 75 %.

**F — Quellen gegeneinander halten (mittel).**
Abschnitt 5 dokumentiert zwei widersprüchliche `id2label`-Tabellen für dieselben
Gewichte, Abschnitt 3 einen Widerspruch zwischen Konfigurationsdatei und
Beispielcode. Formulieren Sie eine Regel, nach der Sie in solchen Fällen entscheiden,
und begründen Sie, warum „das offiziellere Repo gewinnt" keine brauchbare Regel ist.